In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

df['Delivery_Time'].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({'Delivery_Time'})")
plt.xlabel('Delivery_Time')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()


In [ ]:
# Task 1: Write your code here:
df =df.drop('Order_ID', axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)




In [ ]:
col_to_clean = ['Courier_Experience_yrs', 'Delivery_Time']
df_clean = df.dropna(subset='Delivery_Time').copy()
print(f"After dropping missing values: {df_clean.shape}")

df_clean['Weather'] = df_clean['Weather'].fillna('unknown')
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna('unknown')
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna('unknown')
df_clean.head()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

encoder = LabelEncoder()
df_clean['Weather'] = encoder.fit_transform(df_clean['Weather'])
df_clean['Traffic_Level'] = encoder.fit_transform(df_clean['Traffic_Level'])
df_clean['Time_of_Day'] = encoder.fit_transform(df_clean['Time_of_Day'])
df_clean['Vehicle_Type'] = encoder.fit_transform(df_clean['Vehicle_Type'])
df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()


In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)



In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
sklearn_models = {
    "Random Forest": RandomForestRegressor(
      n_estimators=320,  # Number of trees
      max_depth=4)
}

In [ ]:
# Task 2,3,4,5: Write your code here:

result = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate


  mae = mean_absolute_error(y_test, y_pred)
  result.append(mae)

print("Avg score: ",np.mean(result))

In [ ]:
# Task 1: Write your code here:
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_


# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.hist(y_pred, bins=30, edgecolor='black')

plt.title(f"Target Distribution ({y_pred})")
plt.xlabel('Delivery_Time')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
# Task Bonus: Write your code here:

